<a href="https://colab.research.google.com/github/samuelrossiello/data-analytics-portfolio/blob/main/apps/construction_bid_optimizer_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pulp
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
print("Libraries loaded successfully!")

In [ ]:
# ---- PROJECT PARAMETERS ----
trades = [
    "Concrete", "Masonry", "Steel", "Carpentry", "Roofing",
    "Plumbing", "HVAC", "Electrical", "Elevators", "Finishes"
]

base_costs = {
    "Concrete": 63, "Masonry": 27, "Steel": 18, "Carpentry": 33,
    "Roofing": 10, "Plumbing": 27, "HVAC": 30, "Electrical": 34,
    "Elevators": 7, "Finishes": 24
}

gsf = 500000
mbe_goal = 0.20
wbe_goal = 0.10

cert_types = [
    "Non-MWBE", "Non-MWBE", "Non-MWBE",
    "MBE", "WBE", "MWBE"
]

total_budget = sum(base_costs.values()) * gsf
mbe_target = total_budget * mbe_goal
wbe_target = total_budget * wbe_goal

# ---- BID GENERATION ----
def generate_bids():
    rows = []
    for trade in trades:
        base = base_costs[trade] * gsf
        for i, cert in enumerate(cert_types):
            if cert == "Non-MWBE":
                price = base * np.random.uniform(0.90, 1.10)
            elif cert == "MBE":
                price = base * np.random.uniform(0.90, 1.10) * np.random.uniform(1.08, 1.12)
            elif cert == "WBE":
                price = base * np.random.uniform(0.90, 1.10) * np.random.uniform(1.08, 1.12)
            else:
                price = base * np.random.uniform(0.90, 1.10) * np.random.uniform(1.10, 1.15)
            rows.append({
                "trade": trade,
                "bidder_id": f"{trade[:4].upper()}-{i+1}",
                "certification": cert,
                "bid_amount": round(price, 2)
            })
    return pd.DataFrame(rows)

np.random.seed(42)
df_bids = generate_bids()

print(f"Parameters loaded!")
print(f"Total base budget: ${total_budget:,}")
print(f"MBE target: ${mbe_target:,}")
print(f"WBE target: ${wbe_target:,}")
print(f"Bids generated: {len(df_bids)}")

In [ ]:
# ---- OPTIMIZATION ENGINE ----
def run_optimizer(df_bids, total_budget, mbe_target, wbe_target, forced_awards={}):

    # Initialize problem
    prob = pulp.LpProblem("MWBE_Buyout_Optimizer", pulp.LpMinimize)

    # Decision variables
    x = {}
    for _, row in df_bids.iterrows():
        if row["certification"] != "MWBE":
            x[row["bidder_id"], row["trade"]] = pulp.LpVariable(
                f"x_{row['bidder_id']}_{row['trade']}".replace("-", "_"),
                cat="Binary"
            )

    x_mbe = {}
    x_wbe = {}
    for _, row in df_bids[df_bids["certification"] == "MWBE"].iterrows():
        x_mbe[row["bidder_id"], row["trade"]] = pulp.LpVariable(
            f"xmbe_{row['bidder_id']}_{row['trade']}".replace("-", "_"),
            cat="Binary"
        )
        x_wbe[row["bidder_id"], row["trade"]] = pulp.LpVariable(
            f"xwbe_{row['bidder_id']}_{row['trade']}".replace("-", "_"),
            cat="Binary"
        )

    # Objective function
    prob += pulp.lpSum([
        row["bid_amount"] * x[row["bidder_id"], row["trade"]]
        for _, row in df_bids[df_bids["certification"] != "MWBE"].iterrows()
    ] + [
        row["bid_amount"] * x_mbe[row["bidder_id"], row["trade"]] +
        row["bid_amount"] * x_wbe[row["bidder_id"], row["trade"]]
        for _, row in df_bids[df_bids["certification"] == "MWBE"].iterrows()
    ])

    # Constraint 1: One bidder per trade
    for trade in trades:
        trade_bids = df_bids[df_bids["trade"] == trade]
        prob += pulp.lpSum([
            x[row["bidder_id"], row["trade"]]
            for _, row in trade_bids[trade_bids["certification"] != "MWBE"].iterrows()
        ] + [
            x_mbe[row["bidder_id"], row["trade"]] +
            x_wbe[row["bidder_id"], row["trade"]]
            for _, row in trade_bids[trade_bids["certification"] == "MWBE"].iterrows()
        ]) == 1, f"one_bidder_{trade}"

    # Constraint 2: Dual certified MWBE one bucket only
    for _, row in df_bids[df_bids["certification"] == "MWBE"].iterrows():
        prob += x_mbe[row["bidder_id"], row["trade"]] + \
                x_wbe[row["bidder_id"], row["trade"]] <= 1, \
                f"one_bucket_{row['bidder_id']}_{row['trade']}".replace("-", "_")

    # Constraint 3: MBE target
    prob += pulp.lpSum([
        row["bid_amount"] * x[row["bidder_id"], row["trade"]]
        for _, row in df_bids[df_bids["certification"] == "MBE"].iterrows()
    ] + [
        row["bid_amount"] * x_mbe[row["bidder_id"], row["trade"]]
        for _, row in df_bids[df_bids["certification"] == "MWBE"].iterrows()
    ]) >= mbe_target, "mbe_goal"

    # Constraint 4: WBE target
    prob += pulp.lpSum([
        row["bid_amount"] * x[row["bidder_id"], row["trade"]]
        for _, row in df_bids[df_bids["certification"] == "WBE"].iterrows()
    ] + [
        row["bid_amount"] * x_wbe[row["bidder_id"], row["trade"]]
        for _, row in df_bids[df_bids["certification"] == "MWBE"].iterrows()
    ]) >= wbe_target, "wbe_goal"

    # Forced awards — lock in preferred subcontractors
    for trade, bidder_id in forced_awards.items():
        if bidder_id != "None":
            # Check if forced bidder is dual certified MWBE
            cert = df_bids[(df_bids["trade"] == trade) &
                          (df_bids["bidder_id"] == bidder_id)]["certification"].values[0]
            if cert == "MWBE":
                # Force either x_mbe or x_wbe to sum to 1
                prob += x_mbe[bidder_id, trade] + \
                        x_wbe[bidder_id, trade] == 1, \
                        f"forced_{trade}"
            else:
                prob += x[bidder_id, trade] == 1, f"forced_{trade}"

    # Solve
    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    # Extract results
    results = []
    for _, row in df_bids[df_bids["certification"] != "MWBE"].iterrows():
        var = x[row["bidder_id"], row["trade"]]
        if pulp.value(var) == 1:
            results.append({
                "trade": row["trade"],
                "bidder_id": row["bidder_id"],
                "certification": row["certification"],
                "bucket": row["certification"] if row["certification"] != "Non-MWBE" else "Non-MWBE",
                "bid_amount": row["bid_amount"]
            })

    for _, row in df_bids[df_bids["certification"] == "MWBE"].iterrows():
        mbe_val = pulp.value(x_mbe[row["bidder_id"], row["trade"]])
        wbe_val = pulp.value(x_wbe[row["bidder_id"], row["trade"]])
        if mbe_val == 1:
            results.append({
                "trade": row["trade"],
                "bidder_id": row["bidder_id"],
                "certification": row["certification"],
                "bucket": "MBE",
                "bid_amount": row["bid_amount"]
            })
        elif wbe_val == 1:
            results.append({
                "trade": row["trade"],
                "bidder_id": row["bidder_id"],
                "certification": row["certification"],
                "bucket": "WBE",
                "bid_amount": row["bid_amount"]
            })

    df_results = pd.DataFrame(results)
    df_results["trade"] = pd.Categorical(df_results["trade"],
                                          categories=trades, ordered=True)
    df_results = df_results.sort_values("trade").reset_index(drop=True)

    status = pulp.LpStatus[prob.status]
    total_cost = pulp.value(prob.objective)

    return df_results, total_cost, status

print("Optimization engine ready!")

In [ ]:
# ---- WIDGETS ----

# Project parameter inputs
gsf_input = widgets.IntText(
    value=500000,
    description="GSF:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="300px")
)

mbe_input = widgets.FloatText(
    value=20.0,
    description="MBE Goal (%):",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="300px")
)

wbe_input = widgets.FloatText(
    value=10.0,
    description="WBE Goal (%):",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="300px")
)

# Preferred subcontractor dropdowns — one per trade
trade_dropdowns = {}
for trade in trades:
    trade_bids = df_bids[df_bids["trade"] == trade]
    options = ["None (Let optimizer decide)"] + [
        f"{row['bidder_id']} ({row['certification']}) — ${row['bid_amount']:,.0f}"
        for _, row in trade_bids.iterrows()
    ]
    trade_dropdowns[trade] = widgets.Dropdown(
        options=options,
        value="None (Let optimizer decide)",
        description=f"{trade}:",
        style={"description_width": "150px"},
        layout=widgets.Layout(width="500px")
    )

# Buttons
run_button = widgets.Button(
    description="Run Optimizer",
    button_style="success",
    layout=widgets.Layout(width="200px")
)

reset_button = widgets.Button(
    description="Reset All",
    button_style="warning",
    layout=widgets.Layout(width="200px")
)

# Output area
output = widgets.Output()

print("Widgets created!")

In [ ]:
# ---- GREEDY FUNCTION ----
def run_greedy(df_bids, total_budget, mbe_target, wbe_target):

    # Step 1 — Start with lowest bid per trade
    scenario = []
    for trade in trades:
        trade_bids = df_bids[df_bids["trade"] == trade]
        lowest_bid = trade_bids.loc[trade_bids["bid_amount"].idxmin()]
        scenario.append({
            "trade": lowest_bid["trade"],
            "bidder_id": lowest_bid["bidder_id"],
            "certification": lowest_bid["certification"],
            "bucket": "MBE" if lowest_bid["certification"] == "MBE"
                      else "WBE" if lowest_bid["certification"] == "WBE"
                      else "Non-MWBE",
            "bid_amount": lowest_bid["bid_amount"]
        })

    df_greedy = pd.DataFrame(scenario)

    # Step 2 — Swap in cheapest MBE bids until MBE target met
    mbe_spend = df_greedy[df_greedy["bucket"] == "MBE"]["bid_amount"].sum()
    if mbe_spend < mbe_target:
        for trade in trades:
            if mbe_spend >= mbe_target:
                break
            current = df_greedy[df_greedy["trade"] == trade].iloc[0]
            if current["bucket"] != "MBE":
                trade_bids = df_bids[df_bids["trade"] == trade]
                mbe_bids = trade_bids[trade_bids["certification"].isin(["MBE", "MWBE"])]
                if not mbe_bids.empty:
                    cheapest_mbe = mbe_bids.loc[mbe_bids["bid_amount"].idxmin()]
                    df_greedy.loc[df_greedy["trade"] == trade,
                                  ["bidder_id", "certification",
                                   "bucket", "bid_amount"]] = [
                        cheapest_mbe["bidder_id"],
                        cheapest_mbe["certification"],
                        "MBE",
                        cheapest_mbe["bid_amount"]
                    ]
                    mbe_spend = df_greedy[df_greedy["bucket"] == "MBE"]["bid_amount"].sum()

    # Step 3 — Swap in cheapest WBE bids until WBE target met
    wbe_spend = df_greedy[df_greedy["bucket"] == "WBE"]["bid_amount"].sum()
    if wbe_spend < wbe_target:
        for trade in trades:
            if wbe_spend >= wbe_target:
                break
            current = df_greedy[df_greedy["trade"] == trade].iloc[0]
            if current["bucket"] != "WBE" and current["bucket"] != "MBE":
                trade_bids = df_bids[df_bids["trade"] == trade]
                wbe_bids = trade_bids[trade_bids["certification"].isin(["WBE", "MWBE"])]
                if not wbe_bids.empty:
                    cheapest_wbe = wbe_bids.loc[wbe_bids["bid_amount"].idxmin()]
                    df_greedy.loc[df_greedy["trade"] == trade,
                                  ["bidder_id", "certification",
                                   "bucket", "bid_amount"]] = [
                        cheapest_wbe["bidder_id"],
                        cheapest_wbe["certification"],
                        "WBE",
                        cheapest_wbe["bid_amount"]
                    ]
                    wbe_spend = df_greedy[df_greedy["bucket"] == "WBE"]["bid_amount"].sum()

    # Sort and calculate totals
    df_greedy["trade"] = pd.Categorical(df_greedy["trade"],
                                         categories=trades, ordered=True)
    df_greedy = df_greedy.sort_values("trade").reset_index(drop=True)

    greedy_total = df_greedy["bid_amount"].sum()
    greedy_mbe = df_greedy[df_greedy["bucket"] == "MBE"]["bid_amount"].sum()
    greedy_wbe = df_greedy[df_greedy["bucket"] == "WBE"]["bid_amount"].sum()
    greedy_mbe_compliant = greedy_mbe >= mbe_target
    greedy_wbe_compliant = greedy_wbe >= wbe_target

    return df_greedy, greedy_total, greedy_mbe, greedy_wbe, greedy_mbe_compliant, greedy_wbe_compliant

print("Greedy function ready!")

In [ ]:
def run_lowest_bid(df_bids):
    results = []
    for trade in trades:
        trade_bids = df_bids[df_bids["trade"] == trade]
        lowest = trade_bids.loc[trade_bids["bid_amount"].idxmin()]
        results.append({
            "trade": lowest["trade"],
            "bidder_id": lowest["bidder_id"],
            "certification": lowest["certification"],
            "bucket": "Non-MWBE",
            "bid_amount": lowest["bid_amount"]
        })
    df_lowest = pd.DataFrame(results)
    df_lowest["trade"] = pd.Categorical(df_lowest["trade"],
                                         categories=trades, ordered=True)
    df_lowest = df_lowest.sort_values("trade").reset_index(drop=True)
    lowest_total = df_lowest["bid_amount"].sum()
    return df_lowest, lowest_total

print("Lowest bid function ready!")

In [ ]:
def generate_bids_for_gsf(gsf_value):
    rows = []
    for trade in trades:
        base = base_costs[trade] * gsf_value
        for i, cert in enumerate(cert_types):
            if cert == "Non-MWBE":
                price = base * np.random.uniform(0.90, 1.10)
            elif cert == "MBE":
                price = base * np.random.uniform(0.90, 1.10) * np.random.uniform(1.08, 1.12)
            elif cert == "WBE":
                price = base * np.random.uniform(0.90, 1.10) * np.random.uniform(1.08, 1.12)
            else:
                price = base * np.random.uniform(0.90, 1.10) * np.random.uniform(1.10, 1.15)
            rows.append({
                "trade": trade,
                "bidder_id": f"{trade[:4].upper()}-{i+1}",
                "certification": cert,
                "bid_amount": round(price, 2)
            })
    return pd.DataFrame(rows)

print("Helper function ready!")

In [ ]:
# ---- BUTTON CLICK HANDLER ----
def on_run_clicked(b):
    with output:
        clear_output(wait=True)

        print("Running optimizer...")

        # Step 1 — Read project parameters from widgets
        current_gsf = gsf_input.value
        current_mbe_goal = mbe_input.value / 100
        current_wbe_goal = wbe_input.value / 100
        current_budget = sum(base_costs.values()) * current_gsf
        current_mbe_target = current_budget * current_mbe_goal
        current_wbe_target = current_budget * current_wbe_goal

        # Step 2 — Regenerate bids based on current GSF
        np.random.seed(42)
        current_bids = generate_bids_for_gsf(current_gsf)

        # Step 3 — Build forced awards dictionary from dropdowns
        forced_awards = {}
        for trade, dropdown in trade_dropdowns.items():
            selected = dropdown.value
            if selected != "None (Let optimizer decide)":
                # Extract just the bidder_id from the dropdown string
                # Format is "CONC-1 (Non-MWBE) — $30,709,603"
                bidder_id = selected.split(" ")[0]
                forced_awards[trade] = bidder_id

        # Step 4 — Run pure optimal (no forced awards)
        df_optimal, optimal_cost, status = run_optimizer(
            current_bids, current_budget,
            current_mbe_target, current_wbe_target,
            forced_awards={}
        )

        # Step 5 — Run greedy
        df_greedy, greedy_total, greedy_mbe, greedy_wbe, \
        greedy_mbe_compliant, greedy_wbe_compliant = run_greedy(
            current_bids, current_budget,
            current_mbe_target, current_wbe_target
        )

        # Step 5b — Run lowest bid (non-compliant baseline)
        df_lowest, lowest_total = run_lowest_bid(current_bids)

        # Step 6 — Run with forced awards if any exist
        if forced_awards:
            df_forced, forced_cost, forced_status = run_optimizer(
                current_bids, current_budget,
                current_mbe_target, current_wbe_target,
                forced_awards=forced_awards
            )
        else:
            df_forced = None
            forced_cost = None

        display_results(
            df_optimal, optimal_cost,
            df_greedy, greedy_total, greedy_mbe, greedy_wbe,
            greedy_mbe_compliant, greedy_wbe_compliant,
            df_forced, forced_cost,
            forced_awards, current_budget,
            current_mbe_target, current_wbe_target,
            lowest_total
        )

def on_reset_clicked(b):
    with output:
        clear_output(wait=True)
        # Reset all dropdowns to default
        for dropdown in trade_dropdowns.values():
            dropdown.value = "None (Let optimizer decide)"
        # Reset parameters to default
        gsf_input.value = 500000
        mbe_input.value = 20.0
        wbe_input.value = 10.0
        print("Reset complete — adjust parameters and click Run Optimizer")

# Connect buttons to handlers
run_button.on_click(on_run_clicked)
reset_button.on_click(on_reset_clicked)

print("Button handlers connected!")

In [ ]:
def display_results(df_optimal, optimal_cost,
                    df_greedy, greedy_total, greedy_mbe, greedy_wbe,
                    greedy_mbe_compliant, greedy_wbe_compliant,
                    df_forced, forced_cost,
                    forced_awards, current_budget,
                    current_mbe_target, current_wbe_target,
                    lowest_total):

    # ---- HELPER: calculate MWBE totals from results dataframe ----
    def get_mwbe_totals(df):
        mbe = df[df["bucket"] == "MBE"]["bid_amount"].sum()
        wbe = df[df["bucket"] == "WBE"]["bid_amount"].sum()
        return mbe, wbe

    # ---- SECTION 1: PROJECT PARAMETERS ----
    print(f"{'='*65}")
    print(f"CONSTRUCTION BID OPTIMIZER — RESULTS")
    print(f"{'='*65}")
    print(f"Base Budget:    ${current_budget:>15,.0f}")
    print(f"MBE Target:     ${current_mbe_target:>15,.0f} ({current_mbe_target/current_budget*100:.1f}% of budget)")
    print(f"WBE Target:     ${current_wbe_target:>15,.0f} ({current_wbe_target/current_budget*100:.1f}% of budget)")
    if forced_awards:
        print(f"Locked Trades:  {', '.join([f'{t} → {b}' for t, b in forced_awards.items()])}")
    print(f"{'='*65}")

    # ---- SECTION 2: GREEDY RESULTS ----
    print(f"\nSECTION 1 — GREEDY STRATEGY (what a human estimator would do)")
    print(f"{'─'*65}")
    print(df_greedy.to_string(index=False))
    print(f"{'─'*65}")
    print(f"TOTAL BUYOUT COST:    ${greedy_total:>15,.0f}")
    print(f"BASE BUDGET:          ${current_budget:>15,.0f}")
    print(f"SAVINGS:              ${current_budget - greedy_total:>15,.0f}")
    print(f"{'─'*65}")
    greedy_mbe_pct = greedy_mbe / current_budget * 100
    greedy_wbe_pct = greedy_wbe / current_budget * 100
    print(f"MBE SPEND:            ${greedy_mbe:>15,.0f} ({greedy_mbe_pct:.1f}% of budget)")
    print(f"WBE SPEND:            ${greedy_wbe:>15,.0f} ({greedy_wbe_pct:.1f}% of budget)")
    print(f"MBE GOAL MET:         {'YES ✓' if greedy_mbe_compliant else 'NO ✗'}")
    print(f"WBE GOAL MET:         {'YES ✓' if greedy_wbe_compliant else 'NO ✗'}")

    # ---- SECTION 3: PURE OPTIMAL RESULTS ----
    optimal_mbe, optimal_wbe = get_mwbe_totals(df_optimal)
    optimal_mbe_pct = optimal_mbe / current_budget * 100
    optimal_wbe_pct = optimal_wbe / current_budget * 100
    optimal_mbe_compliant = optimal_mbe >= current_mbe_target
    optimal_wbe_compliant = optimal_wbe >= current_wbe_target

    print(f"\nSECTION 2 — PURE OPTIMAL (PuLP — no preferences locked in)")
    print(f"{'─'*65}")

    # Add selection label column
    df_optimal_display = df_optimal.copy()
    df_optimal_display["selection"] = "🤖 Optimizer"
    print(df_optimal_display.to_string(index=False))
    print(f"{'─'*65}")
    print(f"TOTAL BUYOUT COST:    ${optimal_cost:>15,.0f}")
    print(f"BASE BUDGET:          ${current_budget:>15,.0f}")
    print(f"SAVINGS:              ${current_budget - optimal_cost:>15,.0f}")
    print(f"{'─'*65}")
    print(f"MBE SPEND:            ${optimal_mbe:>15,.0f} ({optimal_mbe_pct:.1f}% of budget)")
    print(f"WBE SPEND:            ${optimal_wbe:>15,.0f} ({optimal_wbe_pct:.1f}% of budget)")
    print(f"MBE GOAL MET:         {'YES ✓' if optimal_mbe_compliant else 'NO ✗'}")
    print(f"WBE GOAL MET:         {'YES ✓' if optimal_wbe_compliant else 'NO ✗'}")
    print(f"{'─'*65}")
    print(f"OPTIMIZER VS GREEDY:  ${greedy_total - optimal_cost:>15,.0f} saved by using optimizer")

    # ---- SECTION 4: FORCED SELECTION RESULTS (only if preferences exist) ----
    if df_forced is not None:
        forced_mbe, forced_wbe = get_mwbe_totals(df_forced)
        forced_mbe_pct = forced_mbe / current_budget * 100
        forced_wbe_pct = forced_wbe / current_budget * 100
        forced_mbe_compliant = forced_mbe >= current_mbe_target
        forced_wbe_compliant = forced_wbe >= current_wbe_target

        print(f"\nSECTION 3 — YOUR SELECTION (preferences locked in)")
        print(f"{'─'*65}")

        # Add selection label column — flag forced trades with star
        df_forced_display = df_forced.copy()
        df_forced_display["selection"] = df_forced_display["trade"].apply(
            lambda t: "⭐ Preferred" if t in forced_awards else "🤖 Optimizer"
        )
        print(df_forced_display.to_string(index=False))
        print(f"{'─'*65}")
        print(f"TOTAL BUYOUT COST:    ${forced_cost:>15,.0f}")
        print(f"BASE BUDGET:          ${current_budget:>15,.0f}")
        print(f"SAVINGS:              ${current_budget - forced_cost:>15,.0f}")
        print(f"{'─'*65}")
        print(f"MBE SPEND:            ${forced_mbe:>15,.0f} ({forced_mbe_pct:.1f}% of budget)")
        print(f"WBE SPEND:            ${forced_wbe:>15,.0f} ({forced_wbe_pct:.1f}% of budget)")
        print(f"MBE GOAL MET:         {'YES ✓' if forced_mbe_compliant else 'NO ✗'}")
        print(f"WBE GOAL MET:         {'YES ✓' if forced_wbe_compliant else 'NO ✗'}")
        print(f"{'─'*65}")
        print(f"COST OF PREFERENCES:  ${forced_cost - optimal_cost:>15,.0f} above pure optimal")

        # Value vs greedy
        vs_greedy = greedy_total - forced_cost
        if vs_greedy >= 0:
            print(f"YOUR SELECTION VS GREEDY: ${vs_greedy:>12,.0f} saved vs greedy")
        else:
            print(f"YOUR SELECTION VS GREEDY: ${abs(vs_greedy):>12,.0f} MORE than greedy ⚠️")

    # ---- SECTION 5: SUMMARY COMPARISON ----
    print(f"\n{'='*65}")
    print(f"SUMMARY COMPARISON")
    print(f"{'='*65}")
    print(f"{'Strategy':<35} {'Total Cost':>15} {'Savings':>12}")
    print(f"{'-'*65}")
    print(f"{'Greedy Lowest Bid (non-compliant)':<35} ${lowest_total:>14,.0f} ${current_budget - lowest_total:>11,.0f}")
    print(f"{'Greedy MWBE':<35} ${greedy_total:>14,.0f} ${current_budget - greedy_total:>11,.0f}")
    print(f"{'Pure Optimal (PuLP)':<35} ${optimal_cost:>14,.0f} ${current_budget - optimal_cost:>11,.0f}")
    if df_forced is not None:
      print(f"{'Your Selection':<35} ${forced_cost:>14,.0f} ${current_budget - forced_cost:>11,.0f}")
    print(f"{'='*65}")
    print(f"\nMWBE COMPLIANCE")
    print(f"{'='*65}")
    print(f"{'Strategy':<35} {'MBE %':>8} {'WBE %':>8} {'Compliant':>10}")
    print(f"{'-'*65}")
    print(f"{'Greedy Lowest Bid (non-compliant)':<35} {'0.0%':>8} {'0.0%':>8} {'NO ✗':>10}")
    print(f"{'Greedy MWBE':<35} {greedy_mbe_pct:>7.1f}% {greedy_wbe_pct:>7.1f}% {'YES ✓' if greedy_mbe_compliant and greedy_wbe_compliant else 'NO ✗':>10}")
    print(f"{'Pure Optimal (PuLP)':<35} {optimal_mbe_pct:>7.1f}% {optimal_wbe_pct:>7.1f}% {'YES ✓' if optimal_mbe_compliant and optimal_wbe_compliant else 'NO ✗':>10}")
    if df_forced is not None:
        print(f"{'Your Selection':<35} {forced_mbe_pct:>7.1f}% {forced_wbe_pct:>7.1f}% {'YES ✓' if forced_mbe_compliant and forced_wbe_compliant else 'NO ✗':>10}")
    print(f"{'='*65}")

# ---- VALUE OF OPTIMIZATION ----
    print(f"\nVALUE OF OPTIMIZATION")
    print(f"{'='*65}")
    print(f"Pure Optimal vs Greedy MWBE:  ${greedy_total - optimal_cost:>12,.0f} saved by using optimizer")
    print(f"Cost of MWBE Compliance:      ${optimal_cost - lowest_total:>12,.0f} above non-compliant baseline")
    if df_forced is not None:
        vs_greedy = greedy_total - forced_cost
        if vs_greedy >= 0:
            print(f"Your Selection vs Greedy:     ${vs_greedy:>12,.0f} saved vs greedy approach")
        else:
            print(f"Your Selection vs Greedy:     ${abs(vs_greedy):>12,.0f} MORE expensive than greedy")
        print(f"Cost of Your Preferences:     ${forced_cost - optimal_cost:>12,.0f} above pure optimal")
    print(f"{'='*65}")

    # ---- SECTION 6: WARNING / CAUTION / CONFIRMATION ----
    if df_forced is not None:
        print(f"\nPREFERENCE ANALYSIS")
        print(f"{'='*65}")

        vs_greedy = greedy_total - forced_cost
        cost_of_prefs = forced_cost - optimal_cost

        if vs_greedy < 0:
            # Forced selection costs more than greedy — red warning
            print(f"⚠️  WARNING: Your preferred selections exceed the greedy")
            print(f"    strategy cost by ${abs(vs_greedy):,.0f}.")
            print(f"    Consider releasing some preferences to improve")
            print(f"    cost efficiency.")
        elif cost_of_prefs > optimal_cost * 0.01:
            # More than 1% above pure optimal — yellow caution
            print(f"⚠️  CAUTION: Your preferred selections are ${cost_of_prefs:,.0f}")
            print(f"    above the pure optimal solution (more than 1%).")
            print(f"    Minor adjustments could improve cost efficiency.")
        else:
            # Within 1% of pure optimal — green confirmation
            print(f"✅  GOOD: Your selections are within 1% of the pure")
            print(f"    optimal solution. Good balance of cost efficiency")
            print(f"    and relationship management.")
        print(f"{'='*65}")

        # ---- SECTION 7: VISUALIZATION ----
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Build scenario lists dynamically based on what exists
    scenario_labels = ["Greedy\nLowest Bid\n(Non-Compliant)",
                       "Greedy\nLowest MWBE",
                       "Pure\nOptimal"]
    costs = [lowest_total, greedy_total, optimal_cost]
    mbe_pcts = [0.0, greedy_mbe_pct, optimal_mbe_pct]
    wbe_pcts = [0.0, greedy_wbe_pct, optimal_wbe_pct]
    colors = ["red", "orange", "green"]

    if df_forced is not None:
        scenario_labels.append("Your\nSelection")
        costs.append(forced_cost)
        mbe_pcts.append(forced_mbe_pct)
        wbe_pcts.append(forced_wbe_pct)
        colors.append("blue")

    # Chart 1 - Total cost comparison
    bars = axes[0].bar(scenario_labels, costs, color=colors, alpha=0.7)
    axes[0].axhline(y=current_budget, color="black", linestyle="--",
                    label="Base Budget")
    axes[0].set_title("Total Buyout Cost by Strategy")
    axes[0].set_ylabel("Total Cost")
    axes[0].yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, p: f"${x/1e6:.0f}M"))
    axes[0].legend()
    for bar, cost in zip(bars, costs):
        axes[0].text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + current_budget * 0.003,
                    f"${cost/1e6:.1f}M", ha="center", va="bottom",
                    fontweight="bold", fontsize=8)

    # Chart 2 - MBE compliance
    bars2 = axes[1].bar(scenario_labels, mbe_pcts, color=colors, alpha=0.7)
    axes[1].axhline(y=current_mbe_target/current_budget*100,
                    color="black", linestyle="--",
                    label=f"MBE Goal ({current_mbe_target/current_budget*100:.0f}%)")
    axes[1].set_title("MBE Spend % of Base Budget")
    axes[1].set_ylabel("MBE %")
    axes[1].legend()
    for bar, pct in zip(bars2, mbe_pcts):
        axes[1].text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.3,
                    f"{pct:.1f}%", ha="center", va="bottom",
                    fontweight="bold", fontsize=8)

    # Chart 3 - WBE compliance
    bars3 = axes[2].bar(scenario_labels, wbe_pcts, color=colors, alpha=0.7)
    axes[2].axhline(y=current_wbe_target/current_budget*100,
                    color="black", linestyle="--",
                    label=f"WBE Goal ({current_wbe_target/current_budget*100:.0f}%)")
    axes[2].set_title("WBE Spend % of Base Budget")
    axes[2].set_ylabel("WBE %")
    axes[2].legend()
    for bar, pct in zip(bars3, wbe_pcts):
        axes[2].text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.3,
                    f"{pct:.1f}%", ha="center", va="bottom",
                    fontweight="bold", fontsize=8)

    plt.suptitle("MWBE Buyout Optimization — Strategy Comparison",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print("\nOptimization complete!")

print("display_results() function ready!")

In [ ]:
# ---- APP LAYOUT ----

# Header
print("=" * 65)
print("  CONSTRUCTION BID OPTIMIZER")
print("  MWBE Compliance & Cost Optimization Tool")
print("=" * 65)

# Project parameters section
print("\n📋 PROJECT PARAMETERS")
print("─" * 65)
display(widgets.HBox([gsf_input, mbe_input, wbe_input]))

# Preferred subcontractor section
print("\n🏗️  PREFERRED SUBCONTRACTOR SELECTIONS (optional)")
print("─" * 65)
print("Select preferred subcontractors to lock in — or leave as")
print("'None' to let the optimizer decide for each trade.")
print("─" * 65)

# Display dropdowns in two columns
left_trades = trades[:5]
right_trades = trades[5:]

left_box = widgets.VBox([trade_dropdowns[t] for t in left_trades])
right_box = widgets.VBox([trade_dropdowns[t] for t in right_trades])
display(widgets.HBox([left_box, right_box]))

# Buttons
print("\n")
display(widgets.HBox([run_button, reset_button]))

# Output area
display(output)